<a href="https://colab.research.google.com/github/devaki-turimella/PRODIGY_GI_01/blob/main/Text_Generation_with_GPT_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU, using CPU")


GPU available: False
Device name: No GPU, using CPU


In [ ]:
!pip install -q transformers torch pandas


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"   # small GPT-2 model (124M parameters)

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token by default

model = GPT2LMHeadModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded successfully on:", device)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully on: cpu


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/dev/Quotes.csv", engine='python')

print("Shape of dataset:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape of dataset: (499709, 3)
Columns: ['quote', 'author', 'category']


,quote,author,category
0,"I'm selfish, impatient and a little insecure. ...",Marilyn Monroe,"attributed-no-source, best, life, love, mistak..."
1,You've gotta dance like there's nobody watchin...,William W. Purkey,"dance, heaven, hurt, inspirational, life, love..."
2,You know you're in love when you can't fall as...,Dr. Seuss,"attributed-no-source, dreams, love, reality, s..."
3,A friend is someone who knows all about you an...,Elbert Hubbard,"friend, friendship, knowledge, love"
4,Darkness cannot drive out darkness: only light...,"Martin Luther King Jr., A Testament of Hope: T...","darkness, drive-out, hate, inspirational, ligh..."


In [ ]:
# drop rows where the quote is missing
df = df.dropna(subset=["quote"])

# remove any extra spaces / newlines inside each quote
df["quote"] = df["quote"].astype(str).str.strip()

# taking a random sample of quotes to keep training time reasonable
SAMPLE_SIZE = 3000
df_sample = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# splitting into train (90%) and test (10%)
train_df = df_sample.iloc[: int(0.9 * len(df_sample))]
test_df = df_sample.iloc[int(0.9 * len(df_sample)) :]

print("Train quotes:", len(train_df))
print("Test quotes:", len(test_df))


Train quotes: 2700
Test quotes: 300


In [ ]:
# saving the quotes into text files
# we separate each quote with the GPT-2 end-of-text token so the model
# learns where one quote ends and another begins

def save_quotes_to_file(quote_list, file_path):
    with open(file_path, "w", encoding="utf-8") as f:
        for q in quote_list:
            f.write(q + tokenizer.eos_token + "\n")

train_file = "quotes_train.txt"
test_file = "quotes_test.txt"

save_quotes_to_file(train_df["quote"].tolist(), train_file)
save_quotes_to_file(test_df["quote"].tolist(), test_file)

print("Saved:", train_file, "and", test_file)


Saved: quotes_train.txt and quotes_test.txt


In [ ]:
from torch.utils.data import Dataset

class QuotesDataset(Dataset):
    def __init__(self, file_path, tokenizer, block_size=64):
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        token_ids = tokenizer.encode(text)

        self.examples = []
        for i in range(0, len(token_ids) - block_size + 1, block_size):
            chunk = token_ids[i : i + block_size]
            self.examples.append(chunk)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return torch.tensor(self.examples[idx], dtype=torch.long)


train_dataset = QuotesDataset(train_file, tokenizer, block_size=64)
test_dataset = QuotesDataset(test_file, tokenizer, block_size=64)

print("Train examples:", len(train_dataset))
print("Test examples:", len(test_dataset))


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (127242 > 1024). Running this sequence through the model will result in indexing errors


Train examples: 1988
Test examples: 224


In [ ]:
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   # mlm=False because GPT-2 predicts the next word, not a masked word
)

training_args = TrainingArguments(
    output_dir="gpt2-quotes",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",     # evaluate on test set after every epoch
    save_steps=500,
    logging_steps=20,
    learning_rate=5e-5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.868896,3.759928


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,3.868896,3.759928
2,3.425315,3.784750
3,3.284723,3.820566


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=1491, training_loss=3.542398635690281, metrics={'train_runtime': 6711.8202, 'train_samples_per_second': 0.889, 'train_steps_per_second': 0.222, 'total_flos': 194793209856000.0, 'train_loss': 3.542398635690281, 'epoch': 3.0})

In [ ]:
save_path = "gpt2-quotes-final"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("Model saved at:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved at: gpt2-quotes-final


In [ ]:
def generate_quote(prompt, max_length=40):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    output = model.generate(
        input_ids,
        max_length=max_length,
        do_sample=True,
        top_p=0.9,
        top_k=50,
        temperature=0.8,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)


prompts = ["Once upon a time", "Love is", "The secret to happiness is", "Life is"]

for p in prompts:
    print("Prompt:", p)
    print("Generated:", generate_quote(p))
    print("-" * 80)


Prompt: Once upon a time
Generated: Once upon a time the word of God spread over the whole earth, and it was found in every part of the earth. It was a kind of prophetic news, a warning that our hearts were bound
--------------------------------------------------------------------------------
Prompt: Love is
Generated: Love is the best thing that ever happened to me. And this is why. I was so depressed and so afraid. But I could not help it. It felt like I had gone crazy. Like
--------------------------------------------------------------------------------
Prompt: The secret to happiness is
Generated: The secret to happiness is to love. Love is the joy in all its forms. You will find that the most pleasant of all forms of happiness will be when you love your loved ones. They will
--------------------------------------------------------------------------------
Prompt: Life is
Generated: Life is a part of what we do. Life is our journey, our joy, and our challenge. The road to happiness 

In [ ]:
prompt = "Once upon a time"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

print("Greedy search:")
out = model.generate(input_ids, max_length=40, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True))
print()

print("Beam search:")
out = model.generate(input_ids, max_length=40, num_beams=5, no_repeat_ngram_size=2,
                      early_stopping=True, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True))
print()

print("Top-k sampling:")
out = model.generate(input_ids, max_length=40, do_sample=True, top_k=50,
                      pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True))
print()

print("Top-p sampling:")
out = model.generate(input_ids, max_length=40, do_sample=True, top_p=0.9, top_k=0,
                      pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True))


Greedy search:
Once upon a time, the world was full of people who were not just looking for a place to live, but for a place to work. They were looking for a place to work, and they

Beam search:
Once upon a time there was no such thing as a man without a wife, and it was not until the middle of the eighteenth century that the idea of a woman's existence began to take hold.

Top-k sampling:
Once upon a time of greatest need there is not anything of the sort here with respect to money; such as when it is due to any one man; and such as when it is due to any

Top-p sampling:
Once upon a time there was a wiseman in the room. One of his harps trembled upon one side of his skull; and before his eyes were drooping with anxiety, a vision


In [ ]:
import math

eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)

print("Eval loss:", eval_loss)
print("Perplexity:", perplexity)


Training Loss,Validation Loss,Epoch
3.284723,3.820566,3


Eval loss: 3.820566415786743
Perplexity: 45.6300465812921
